<a href="https://colab.research.google.com/github/pavan-charan/Auth_Module/blob/main/Neuromorphic_Lane_Detection_Review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Real-time & Resource-Efficient Lane Detection — Review Presentation
## CNN Preprocessing vs SNN Preprocessing | Weather Comparison | Parameter Analysis

> **Indian Institute of Information Technology Kottayam** | B.Tech Honours Project  
> **Student:** Siraparapu Sai Satya Pavan Charan (2023BCS0078)  
> **Supervisor:** Dr. Goutam Mali

---

### Presentation Scope

| Section | Content |
|---------|--------|
| **1** | Environment Setup |
| **2** | Dataset Loading & Verification |
| **3** | CNN Preprocessing Pipeline |
| **4** | SNN Preprocessing — Spike Encoding |
| **5** | SNN Spike Parameter Tuning (Grid Search) |
| **6** | CNN vs SNN Preprocessing under Weather Conditions |
| **7** | CNN vs SNN Parameter & Storage Comparison |
| **8** | Complete Visual Summary |


---
## Section 1: Environment Setup

Install all required packages. Run once per Colab session.  
**First:** Runtime → Change runtime type → **T4 GPU**


In [ ]:
# ================================================================
# CELL 1: Package Installation
# ================================================================
import subprocess, sys
packages = ['snntorch', 'albumentations>=1.3.0',
            'opencv-python-headless', 'pandas', 'seaborn']
for pkg in packages:
    r = subprocess.run([sys.executable,'-m','pip','install',pkg,'-q'], capture_output=True)
    print(f'  [{'OK' if r.returncode==0 else 'FAIL'}] {pkg}')
print('All packages ready.')


  [OK] snntorch
  [OK] albumentations>=1.3.0
  [OK] opencv-python-headless
  [OK] pandas
  [OK] seaborn
All packages ready.


---
## Section 2: Imports, Configuration & Dataset Loading


In [ ]:
# ================================================================
# CELL 2: Imports & Device
# ================================================================
import os, sys, time, json, copy, random, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import snntorch as snn
from snntorch import surrogate, utils
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED   = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

print(f'Device  : {DEVICE}')
if torch.cuda.is_available():
    print(f'  GPU   : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print(f'PyTorch : {torch.__version__}')
print(f'snnTorch: {snn.__version__}')


Device  : cuda
  GPU   : Tesla T4
  VRAM  : 15.6 GB
PyTorch : 2.10.0+cu128
snnTorch: 0.9.4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ================================================================
# CELL 3: Configuration
# ================================================================
CFG = {
    'train_img_dir' : '/content/drive/MyDrive/bdd100k_10k_split/images/train',
    'val_img_dir'   : '/content/drive/MyDrive/bdd100k_10k_split/images/val',
    'test_img_dir'  : '/content/drive/MyDrive/bdd100k_10k_split/images/test',
    'train_lbl_dir' : '/content/drive/MyDrive/bdd100k_10k_split/labels/train',
    'val_lbl_dir'   : '/content/drive/MyDrive/bdd100k_10k_split/labels/val',
    'test_lbl_dir'  : '/content/drive/MyDrive/bdd100k_10k_split/labels/test',
    'output_dir'    : '/content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn',
    'img_h'         : 256,
    'img_w'         : 512,
    'max_train'     : 2000,
    'max_val'       : 400,
    'max_test'      : 200,
    'cnn_batch'     : 8,
    'cnn_features'  : [32, 64, 128, 256],
    'cnn_bottleneck': 512,
    'cnn_dropout'   : 0.2,
    'roi_top_frac'  : 0.40,
    # SNN spike encoding defaults (will be tuned in Section 5)
    'T'             : 4,
    'spike_thresh'  : 0.07,
    'noise_std'     : 0.04,
    'beta'          : 0.95,
    'surrogate_slope': 25,
    'eval_thresh'   : 0.5,
}
os.makedirs(CFG['output_dir'], exist_ok=True)
print('CFG loaded. Output dir:', CFG['output_dir'])


CFG loaded. Output dir: /content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn


In [ ]:
# ================================================================
# CELL 4: Mount Google Drive & Verify Dataset
# ================================================================


print('=' * 58)
print(f"  {'Split':<8} {'Images':>8}  {'Labels':>8}  {'Status':>8}")
print('=' * 58)
all_ok = True
for split in ['train', 'val', 'test']:
    idir = CFG[f'{split}_img_dir']
    ldir = CFG[f'{split}_lbl_dir']
    n_i = len([f for f in os.listdir(idir) if f.lower().endswith('.jpg')]) if os.path.isdir(idir) else 0
    n_l = len([f for f in os.listdir(ldir) if f.endswith('.json')])        if os.path.isdir(ldir) else 0
    ok  = n_i > 0 and n_l > 0
    if not ok: all_ok = False
    print(f"  {split:<8} {n_i:>8,}  {n_l:>8,}  {'OK' if ok else 'FAIL':>8}")
print('=' * 58)
print('All files ready.' if all_ok else 'ERROR: check Drive paths in CFG.')


  Split      Images    Labels    Status
  train       7,000     7,000        OK
  val         1,000     1,000        OK
  test        2,000     2,000        OK
All files ready.


In [ ]:
# ================================================================
# CELL 5: Dataset Class & DataLoaders
# ================================================================
class BDD100KLaneDataset(Dataset):
    LANE_THICKNESS = 8
    def __init__(self, img_dir, lbl_dir, transform=None, max_samples=None):
        self.img_dir = Path(img_dir); self.lbl_dir = Path(lbl_dir)
        self.transform = transform; self.samples = []
        for f in sorted(self.img_dir.iterdir()):
            if f.suffix.lower() not in {'.jpg','.jpeg','.png'}: continue
            lp = self.lbl_dir / (f.stem + '.json')
            if lp.exists(): self.samples.append((f, lp))
        if max_samples: self.samples = self.samples[:max_samples]
        print(f'  {len(self.samples):,} samples <- {str(img_dir)[-40:]}')

    def __len__(self): return len(self.samples)

    def _rasterise(self, lbl_data, h, w):
        mask = np.zeros((h, w), np.uint8)
        if 'frames' not in lbl_data: return mask
        frame = lbl_data['frames'][0]
        for obj in frame.get('objects', []):
            if not obj.get('category','').startswith('lane'): continue
            if 'poly2d' not in obj: continue
            pts = np.array([[int(p[0]),int(p[1])] for p in obj['poly2d']], np.int32)
            if len(pts) >= 2:
                cv2.polylines(mask,[pts],False,255,self.LANE_THICKNESS)
        return mask

    def __getitem__(self, idx):
        ip, lp = self.samples[idx]
        img = cv2.imread(str(ip))
        if img is None: img = np.zeros((720,1280,3),np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        with open(lp) as f: ldata = json.load(f)
        mask = self._rasterise(ldata, h, w)
        if self.transform:
            t = self.transform(image=img, mask=mask)
            img = t['image']; mask = t['mask']
        else:
            img  = torch.from_numpy(img.transpose(2,0,1)).float()/255.0
            mask = torch.from_numpy(mask).float()
        return img, (mask>0).float().unsqueeze(0)

def build_val_transforms(h=256, w=512):
    return A.Compose([A.Resize(h,w), A.Normalize(mean=IMAGENET_MEAN,std=IMAGENET_STD), ToTensorV2()],
                     additional_targets={'mask':'mask'})

def build_train_transforms(h=256, w=512):
    return A.Compose([
        A.Resize(h,w),
        A.RandomBrightnessContrast(0.3,0.3,p=0.5),
        A.GaussNoise(var_limit=(10,50),p=0.3),
        A.MotionBlur(blur_limit=7,p=0.2),
        A.RandomRain(slant_lower=-10,slant_upper=10,drop_length=15,drop_width=1,p=0.15),
        A.RandomFog(fog_coef_lower=0.1,fog_coef_upper=0.3,p=0.15),
        A.HorizontalFlip(p=0.5),
        A.Normalize(mean=IMAGENET_MEAN,std=IMAGENET_STD),
        ToTensorV2()
    ], additional_targets={'mask':'mask'})

val_tf = build_val_transforms(CFG['img_h'], CFG['img_w'])
val_ds = BDD100KLaneDataset(CFG['val_img_dir'], CFG['val_lbl_dir'],
                             transform=val_tf, max_samples=CFG['max_val'])
val_loader = DataLoader(val_ds, batch_size=CFG['cnn_batch'], shuffle=False, num_workers=2, pin_memory=True)
print(f'Val loader ready: {len(val_ds):,} samples')


  400 samples <- ive/MyDrive/bdd100k_10k_split/images/val
Val loader ready: 400 samples


---
## Section 3: CNN Preprocessing Pipeline

The CNN receives a dense RGB image. Every pixel is processed at every frame.
The preprocessing chain normalises the input so the U-Net learns consistently.

| Stage | Operation | Purpose |
|-------|-----------|--------|
| 1 | Gaussian Blur | Suppress sensor noise — reduces false gradients |
| 2 | CLAHE | Contrast normalisation — day/night/fog consistency |
| 3 | ROI Mask | Zero out top 40% (sky) — remove non-road pixels |
| 4 | Resize 256×512 | Fixed resolution for batch computation |
| 5 | Normalise | ImageNet mean/std — match CNN weight priors |

**CNN complexity:** $O(n^2 k^2)$ — every pixel computed every frame regardless of change.


In [ ]:
# ================================================================
# CELL 6: CNN Preprocessing Functions + Step-by-Step Visualisation
# ================================================================
def apply_gaussian(img, ksize=5, sigma=1.0):
    return cv2.GaussianBlur(img, (ksize,ksize), sigma)

def apply_clahe(img):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l,a,b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    return cv2.cvtColor(cv2.merge([clahe.apply(l),a,b]), cv2.COLOR_LAB2RGB)

def apply_roi(img, top_frac=0.40):
    m = img.copy(); m[:int(img.shape[0]*top_frac)] = 0; return m

def normalise_img(img):
    m = np.array(IMAGENET_MEAN); s = np.array(IMAGENET_STD)
    return ((img/255.0 - m)/s).clip(-3,3)

# Pick one sample image
img_files = sorted([f for f in os.listdir(CFG['val_img_dir']) if f.endswith('.jpg')])
SAMPLE_PATH = os.path.join(CFG['val_img_dir'], img_files[0])
raw_bgr = cv2.imread(SAMPLE_PATH)
raw_rgb = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)

gauss   = apply_gaussian(raw_rgb)
clahe   = apply_clahe(gauss)
roi     = apply_roi(clahe, CFG['roi_top_frac'])
resized = cv2.resize(roi, (CFG['img_w'], CFG['img_h']))
normed  = normalise_img(resized)

# Step-by-step figure
stages = [raw_rgb, gauss, clahe, roi, resized]
lbls   = ['1. Raw Input\n(1280x720)', '2. Gaussian\nSmoothing', '3. CLAHE\nContrast', '4. ROI Mask\n(top 40% zeroed)', '5. Resized\n256x512']
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
fig.suptitle('CNN Preprocessing Pipeline — Step by Step', fontsize=13, fontweight='bold')
for ax, stage, lbl in zip(axes, stages, lbls):
    ax.imshow(cv2.resize(stage, (256,128)))
    ax.set_title(lbl, fontsize=9, fontweight='bold'); ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(CFG['output_dir'],'P01_cnn_pipeline.png'), dpi=140, bbox_inches='tight')
plt.show(); print('Saved → P01_cnn_pipeline.png')


Saved → P01_cnn_pipeline.png


In [ ]:
# ================================================================
# CELL 7: CNN Preprocessing — Intensity Histograms & ROI Effect
# ================================================================

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('CNN Preprocessing — Detailed Analysis', fontsize=13, fontweight='bold')

# Row 1: before/after images
for ax, img, title in zip(axes[0],
    [raw_rgb, clahe, roi],
    ['Raw Input', 'After CLAHE', 'After ROI Mask']):
    ax.imshow(cv2.resize(img, (256,128)))
    ax.set_title(title, fontweight='bold'); ax.axis('off')

# Row 2: histograms
for ax, img, title, col in zip(axes[1],
    [raw_rgb, clahe, roi],
    ['Raw — Intensity', 'CLAHE — Intensity', 'ROI — Intensity (road only)'],
    ['steelblue','darkorange','green']):
    ax.hist(img[img>0].ravel() if 'ROI' in title else img.ravel(),
            bins=64, color=col, alpha=0.8, edgecolor='none')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Pixel Value'); ax.set_ylabel('Count'); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CFG['output_dir'],'P02_cnn_histograms.png'), dpi=130, bbox_inches='tight')
plt.show(); print('Saved → P02_cnn_histograms.png')

# Road pixel stats
total_px  = raw_rgb.shape[0] * raw_rgb.shape[1]
road_px   = int(total_px * (1 - CFG['roi_top_frac']))
print(f'ROI Analysis:')
print(f'  Total pixels         : {total_px:,}')
print(f'  Road pixels (bottom 60%): {road_px:,}  ({road_px/total_px*100:.0f}%)')
print(f'  Sky pixels zeroed    : {total_px-road_px:,}  ({(total_px-road_px)/total_px*100:.0f}%)')
print(f'  CNN MACs saved by ROI: ~{(total_px-road_px)/total_px*100:.0f}% of non-road computation eliminated')


Saved → P02_cnn_histograms.png
ROI Analysis:
  Total pixels         : 921,600
  Road pixels (bottom 60%): 552,960  (60%)
  Sky pixels zeroed    : 368,640  (40%)
  CNN MACs saved by ROI: ~40% of non-road computation eliminated


In [ ]:
# ================================================================
# CELL 8: CNN Preprocessing — Batch with Augmentation Demo
# Shows: what the CNN actually receives during training
# ================================================================

# Show 4 samples with mask overlay
imgs, masks = next(iter(val_loader))
mean_np = np.array(IMAGENET_MEAN); std_np = np.array(IMAGENET_STD)

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
fig.suptitle('CNN Input Batch — Image / Lane Mask / Overlay (BDD100K)', fontsize=12, fontweight='bold')
axes[0,0].set_ylabel('RGB Input', fontsize=10, fontweight='bold')
axes[1,0].set_ylabel('Lane Mask', fontsize=10, fontweight='bold')
axes[2,0].set_ylabel('Overlay',   fontsize=10, fontweight='bold')

lane_pcts = []
for i in range(4):
    img_np  = (imgs[i].permute(1,2,0).numpy()*std_np+mean_np).clip(0,1)
    msk_np  = masks[i,0].numpy()
    lane_pct = msk_np.mean()*100; lane_pcts.append(lane_pct)
    overlay = img_np.copy(); overlay[msk_np>0.5] = [1.0,0.25,0.0]
    axes[0,i].imshow(img_np); axes[0,i].axis('off')
    axes[0,i].set_title(f'Sample {i+1}', fontsize=9)
    axes[1,i].imshow(msk_np, cmap='hot',vmin=0,vmax=1); axes[1,i].axis('off')
    axes[1,i].set_title(f'Lane: {lane_pct:.2f}%', fontsize=8, color='darkorange')
    axes[2,i].imshow(overlay.clip(0,1)); axes[2,i].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(CFG['output_dir'],'P03_cnn_batch.png'), dpi=130, bbox_inches='tight')
plt.show(); print('Saved → P03_cnn_batch.png')

print(f'Class imbalance: avg lane = {np.mean(lane_pcts):.2f}% of pixels')
print(f'Background dominance: {100-np.mean(lane_pcts):.2f}%  — handled by pos_weight=5.0 in BCE loss')


Saved → P03_cnn_batch.png
Class imbalance: avg lane = 2.11% of pixels
Background dominance: 97.89%  — handled by pos_weight=5.0 in BCE loss


---
## Section 4: SNN Preprocessing — Temporal Difference Spike Encoding

The SNN does **not** receive dense RGB frames. Instead:
1. A pseudo-temporal sequence is created from the static image by adding controlled shifts
2. Frame-to-frame differences are computed
3. Only pixels where change exceeds threshold θ fire a spike

$$\Delta I_t(x,y) = I_t(x,y) - I_{t-1}(x,y)$$

$$S_t(x,y) = \begin{cases}1 & |\Delta I_t| > \theta \\ 0 & \text{otherwise}\end{cases}$$

**Key insight:** Static background pixels ($\Delta I \approx 0$) generate **no spikes** — zero computation on ~90% of pixels. Only lane edges and moving regions fire.

**SNN complexity:** $O(S)$ where $S$ = active spike count $\ll n^2k^2$ (CNN)


In [ ]:
# ================================================================
# CELL 9: SNN Spike Encoding Functions (FINAL VERSION)
# ================================================================

import torch
import torch.nn.functional as F

def make_temporal_sequence(imgs, T=4, noise_std=0.04, mode="shift"):
    """
    Create pseudo-temporal sequence.

    Modes:
    - "noise" → Gaussian noise (original)
    - "shift" → spatial motion (better for demo)
    """

    seq = [imgs]

    for t in range(1, T):
        if mode == "noise":
            new = (imgs + torch.randn_like(imgs) * noise_std).clamp(0, 1)

        elif mode == "shift":
            new = torch.roll(imgs, shifts=(t*2, 0), dims=(2, 3))

        else:
            raise ValueError("mode must be 'noise' or 'shift'")

        seq.append(new)

    return torch.stack(seq, dim=1)


def temporal_difference_encode(img_seq, threshold=None, grayscale=True):
    """
    Temporal Difference Spike Encoding (Adaptive).

    Improvements:
    - Grayscale conversion (reduces redundancy)
    - Smoothing (noise reduction)
    - Normalization (stable range)
    - Adaptive threshold (no manual tuning)

    Returns:
        spikes: (T-1, B, 1, H, W)
    """

    # ------------------ Grayscale ------------------
    if grayscale:
        w = torch.tensor([0.2989, 0.5870, 0.1140],
                         device=img_seq.device).view(1,1,3,1,1)
        img_seq = (img_seq * w).sum(2, keepdim=True)

    spikes = []

    for t in range(1, img_seq.shape[1]):

        # ------------------ Temporal Difference ------------------
        delta = img_seq[:, t] - img_seq[:, t-1]

        # ------------------ Smooth ------------------
        delta = F.avg_pool2d(delta, 3, stride=1, padding=1)

        # ------------------ Normalize ------------------
        delta = delta / (delta.abs().max() + 1e-8)

        # ------------------ Adaptive Threshold ------------------
        if threshold is None:
            mean = delta.abs().mean()
            std  = delta.abs().std()
            theta = mean + 0.5 * std
            theta = torch.clamp(theta, 0.15, 0.4)
        else:
            theta = threshold

        # ------------------ Spike ------------------
        spike = (delta.abs() > theta).float()
        spikes.append(spike)

    return torch.stack(spikes, dim=0)


# ================================================================
# INFO PRINT
# ================================================================

print('✅ Spike encoding functions defined (FINAL VERSION)')
print(f'  Time steps (T)      = {CFG["T"]}')
print(f'  Noise std           = {CFG["noise_std"]}')
print(f'  Threshold           = Adaptive (recommended)')

✅ Spike encoding functions defined (FINAL VERSION)
  Time steps (T)      = 4
  Noise std           = 0.04
  Threshold           = Adaptive (recommended)


In [ ]:
# ================================================================
# CELL 10: FINAL SNN Visualization (Presentation Ready)
# ================================================================

demo_imgs, demo_masks = next(iter(val_loader))
demo_imgs = demo_imgs.to(DEVICE)

with torch.no_grad():

    # ------------------ Temporal Sequence ------------------
    seq = make_temporal_sequence(
        demo_imgs,
        T=CFG['T'],
        mode="shift"
    )

    # ------------------ Spike Encoding ------------------
    spk = temporal_difference_encode(
        seq,
        threshold=None   # adaptive
    )

# ================================================================
# VISUALIZATION
# ================================================================

mean_np = np.array(IMAGENET_MEAN)
std_np  = np.array(IMAGENET_STD)

img_np  = (demo_imgs[0].permute(1,2,0).cpu().numpy()*std_np+mean_np).clip(0,1)
frame1  = (seq[0,0].permute(1,2,0).cpu().numpy()*std_np+mean_np).clip(0,1)
frame2  = (seq[0,1].permute(1,2,0).cpu().numpy()*std_np+mean_np).clip(0,1)

# ------------------ Diff Map ------------------
diff = (seq[:,1] - seq[:,0]).abs()
diff = diff / (diff.max() + 1e-8)
diff_np = diff[0].mean(0).cpu().numpy()

# ------------------ Spike Map ------------------
spk_np = spk[0,0].cpu().numpy()

if spk_np.ndim == 3:
    spk_np = (spk_np.sum(axis=0) > 0).astype(float)

# ------------------ Overlay (GREEN SPIKES) ------------------
spk_overlay = img_np.copy()
spk_overlay[spk_np > 0.5] = [0.0, 1.0, 0.0]

# ================================================================
# PLOTTING
# ================================================================

fig, axes = plt.subplots(1, 5, figsize=(22, 4))

# 🔥 Title with spike percentage
fig.suptitle(
    f'Temporal Difference Spike Encoding — Sparse Event Representation ({spk_np.mean()*100:.2f}% Active)',
    fontsize=14,
    fontweight='bold'
)

# Panels
axes[0].imshow(frame1)
axes[0].set_title('Frame t (Input)', fontsize=10, fontweight='bold')
axes[0].axis('off')

axes[1].imshow(frame2)
axes[1].set_title('Frame t+1 (Shifted)', fontsize=10, fontweight='bold')
axes[1].axis('off')

# 🔥 Diff with colorbar
im = axes[2].imshow(diff_np, cmap='plasma')
axes[2].set_title('Normalized |Δ|', fontsize=10, fontweight='bold')
axes[2].axis('off')
fig.colorbar(im, ax=axes[2], fraction=0.046)

axes[3].imshow(spk_np, cmap='gray')
axes[3].set_title('Spike Map', fontsize=10, fontweight='bold')
axes[3].axis('off')

axes[4].imshow(spk_overlay)
axes[4].set_title('Spike Overlay (Green = Active)', fontsize=10, fontweight='bold')
axes[4].axis('off')

plt.tight_layout()

plt.savefig(
    os.path.join(CFG['output_dir'], 'P04_FINAL_spike_encoding.png'),
    dpi=150,
    bbox_inches='tight'
)

plt.show()
print('Saved → P04_FINAL_spike_encoding.png')

# ================================================================
# STATS
# ================================================================

sr = spk.mean().item()
mac_cnn = CFG['img_h'] * CFG['img_w'] * 9 * 6

print()
print('=' * 50)
print('  SNN Spike Encoding Statistics (FINAL)')
print('=' * 50)
print(f'  Spike train shape : {tuple(spk.shape)}')
print(f'  Spike rate        : {sr*100:.2f}% of pixels fire')
print(f'  Silent pixels     : {(1-sr)*100:.2f}% — zero computation')
print(f'  CNN MACs/frame    : {mac_cnn/1e6:.2f} M (dense)')
print(f'  SNN MACs/frame    : {mac_cnn*sr/1e6:.3f} M (sparse)')
print(f'  MAC reduction     : {(1-sr)*100:.1f}%')
print('=' * 50)

Saved → P04_FINAL_spike_encoding.png

  SNN Spike Encoding Statistics (FINAL)
  Spike train shape : (3, 8, 1, 256, 512)
  Spike rate        : 8.01% of pixels fire
  Silent pixels     : 91.99% — zero computation
  CNN MACs/frame    : 7.08 M (dense)
  SNN MACs/frame    : 0.567 M (sparse)
  MAC reduction     : 92.0%


In [ ]:
# ================================================================
# CELL 11: Save SNN Outputs to Google Drive
# ================================================================

import os

save_dir = "/content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn/snn_outputs"
os.makedirs(save_dir, exist_ok=True)

num_samples = 40
count = 0

for imgs, _ in val_loader:

    imgs = imgs.to(DEVICE)

    with torch.no_grad():
        seq = make_temporal_sequence(imgs, T=CFG['T'], mode="shift")
        spk = temporal_difference_encode(seq, threshold=None)

    for b in range(imgs.size(0)):

        if count >= num_samples:
            break

        # ---------------- IMAGE ----------------
        img_np = (imgs[b].permute(1,2,0).cpu().numpy()*np.array(IMAGENET_STD)+np.array(IMAGENET_MEAN)).clip(0,1)
        frame1 = (seq[b,0].permute(1,2,0).cpu().numpy()*np.array(IMAGENET_STD)+np.array(IMAGENET_MEAN)).clip(0,1)
        frame2 = (seq[b,1].permute(1,2,0).cpu().numpy()*np.array(IMAGENET_STD)+np.array(IMAGENET_MEAN)).clip(0,1)

        # ---------------- DIFF ----------------
        diff = (seq[b,1] - seq[b,0]).abs()
        diff = diff / (diff.max() + 1e-8)
        diff_np = diff.mean(0).cpu().numpy()

        # ---------------- SPIKES ----------------
        spk_np = spk[0,b].cpu().numpy()
        if spk_np.ndim == 3:
            spk_np = (spk_np.sum(axis=0) > 0).astype(float)

        # ---------------- OVERLAY ----------------
        overlay = img_np.copy()
        overlay[spk_np > 0.5] = [0.0, 1.0, 0.0]

        # ---------------- PLOT ----------------
        fig, axes = plt.subplots(1, 5, figsize=(18, 3))

        fig.suptitle(
            f'SNN Spike Encoding ({spk_np.mean()*100:.2f}% Active)',
            fontsize=12, fontweight='bold'
        )

        axes[0].imshow(frame1); axes[0].set_title('Frame t'); axes[0].axis('off')
        axes[1].imshow(frame2); axes[1].set_title('Frame t+1'); axes[1].axis('off')

        im = axes[2].imshow(diff_np, cmap='plasma')
        axes[2].set_title('|Δ|'); axes[2].axis('off')
        fig.colorbar(im, ax=axes[2], fraction=0.046)

        axes[3].imshow(spk_np, cmap='gray')
        axes[3].set_title('Spikes'); axes[3].axis('off')

        axes[4].imshow(overlay)
        axes[4].set_title('Overlay'); axes[4].axis('off')

        plt.tight_layout()

        # ---------------- SAVE TO DRIVE ----------------
        save_path = os.path.join(save_dir, f"snn_{count:03d}.png")
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close()

        print(f"Saved to Drive: {save_path}")

        count += 1

    if count >= num_samples:
        break

print("\n✅ All images saved to Google Drive!")
print(f"📁 Location: {save_dir}")

Saved to Drive: /content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn/snn_outputs/snn_000.png
Saved to Drive: /content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn/snn_outputs/snn_001.png
Saved to Drive: /content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn/snn_outputs/snn_002.png
Saved to Drive: /content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn/snn_outputs/snn_003.png
Saved to Drive: /content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn/snn_outputs/snn_004.png
Saved to Drive: /content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn/snn_outputs/snn_005.png
Saved to Drive: /content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn/snn_outputs/snn_006.png
Saved to Drive: /content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn/snn_outputs/snn_007.png
Saved to Drive: /content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn/snn_outputs/snn_008.png
Saved to Drive: /content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn/snn_outputs/snn_009.png


In [ ]:
# ================================================================
# CELL 11: SNN vs CNN Preprocessing Comparison — Side by Side
# ================================================================

fig = plt.figure(figsize=(20, 9))
fig.suptitle('CNN Preprocessing vs SNN Preprocessing — Key Differences', fontsize=14, fontweight='bold')
gs = gridspec.GridSpec(2, 5, figure=fig, hspace=0.45, wspace=0.1)

# ── CNN Row ──────────────────────────────────────────────────
cnn_stages  = [raw_rgb, gauss, clahe, roi, resized]
cnn_titles  = ['Raw Input', 'Gaussian\nFilter', 'CLAHE\nNormalise', 'ROI Mask\n(sky removed)', 'Final CNN\nInput 256x512']
for col, (stage, title) in enumerate(zip(cnn_stages, cnn_titles)):
    ax = fig.add_subplot(gs[0, col])
    ax.imshow(cv2.resize(stage, (256,128)))
    ax.set_title(title, fontsize=9, fontweight='bold'); ax.axis('off')
    if col == 0: ax.set_ylabel('CNN\nPath', fontsize=11, fontweight='bold', color='steelblue', rotation=90, labelpad=8)

# ── SNN Row ──────────────────────────────────────────────────
snn_panels = [
    (frame1, 'Frame t\n(same as CNN input)', None),
    (diff_np,'|Δ| Difference\n(temporal change)','plasma'),
    (spk_np, f'Spike Map\n(θ={CFG["spike_thresh"]}  rate={spk_np.mean()*100:.1f}%)','hot'),
    (spk_overlay,'Active Spikes\n(red) on scene', None),
]
# blank cell for alignment
ax_blank = fig.add_subplot(gs[1,0])
ax_blank.set_facecolor('#f0f0f0')
ax_blank.text(0.5,0.5,'Grayscale\nConversion\n(3ch → 1ch)\n3× less data',
              ha='center',va='center',fontsize=9,fontweight='bold',color='darkorange',
              transform=ax_blank.transAxes)
ax_blank.axis('off')
ax_blank.set_ylabel('SNN\nPath', fontsize=11, fontweight='bold', color='darkorange', rotation=90, labelpad=8)
for col, (panel, title, cmap) in enumerate(snn_panels, 1):
    ax = fig.add_subplot(gs[1, col])
    ax.imshow(panel.clip(0,1), cmap=cmap); ax.set_title(title, fontsize=9, fontweight='bold'); ax.axis('off')

plt.savefig(os.path.join(CFG['output_dir'],'P05_cnn_vs_snn_preprocessing.png'), dpi=140, bbox_inches='tight')
plt.show(); print('Saved → P05_cnn_vs_snn_preprocessing.png')


Saved → P05_cnn_vs_snn_preprocessing.png


---
## Section 5: SNN Spike Encoding Parameter Tuning

Before training the SNN, the **spike encoding parameters** must be carefully tuned to achieve a 5–15% spike rate. This is the energy-efficient regime.

Too high spike rate (>20%) → LIF neurons saturate → gradient collapse → IoU ≈ 0  
Too low spike rate (<2%) → insufficient information → no learning

### Parameters Being Tuned

| Parameter | Effect | Target |
|-----------|--------|-------|
| **θ (spike_thresh)** | Lower = more spikes | Must find sweet spot 5–15% rate |
| **T (time steps)** | More steps = more temporal context | T=4 to T=8 |
| **noise_std** | Inter-frame variation strength | Must create detectable differences |

**This is a grid search over spike encoding parameters only — no model training required.**


In [ ]:
# ================================================================
# CELL 12: Grid Search (FINAL — Aligned with New Pipeline)
# ================================================================

import pandas as pd
import os

# 🔥 Save results to Google Drive
save_dir = "/content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn/snn_analysis"
os.makedirs(save_dir, exist_ok=True)

# ---------------- SEARCH SPACE ----------------
THETA_VALUES = [0.05, 0.10, 0.15, 0.20, 0.30]
T_VALUES     = [3, 4, 5, 6]
MODES        = ["shift", "noise"]

# ---------------- FIXED BATCH ----------------
ref_imgs, _ = next(iter(val_loader))
ref_imgs = ref_imgs.to(DEVICE)

# ================================================================
# 1. THRESHOLD SWEEP (ONLY for fixed threshold)
# ================================================================

print("🔍 Sweeping threshold θ (fixed mode)...")

theta_results = []

for theta in THETA_VALUES:
    with torch.no_grad():
        seq = make_temporal_sequence(ref_imgs, T=4, mode="shift")
        spk = temporal_difference_encode(seq, threshold=theta)

    sr = spk.mean().item() * 100

    status = "✓ IDEAL" if 5 <= sr <= 15 else ("HIGH" if sr > 15 else "LOW")

    theta_results.append({
        "theta": theta,
        "spike_rate (%)": sr,
        "status": status
    })

    print(f"θ={theta:.2f} → {sr:.2f}% ({status})")

theta_df = pd.DataFrame(theta_results)

# ================================================================
# 2. TIME STEPS SWEEP (Adaptive threshold)
# ================================================================

print("\n🔍 Sweeping time steps T (adaptive)...")

t_results = []

for T in T_VALUES:
    with torch.no_grad():
        seq = make_temporal_sequence(ref_imgs, T=T, mode="shift")
        spk = temporal_difference_encode(seq, threshold=None)

    sr = spk.mean().item() * 100

    t_results.append({
        "T": T,
        "spike_rate (%)": sr,
        "temporal_steps": T-1
    })

    print(f"T={T} → {sr:.2f}%")

t_df = pd.DataFrame(t_results)

# ================================================================
# 3. MODE COMPARISON (SHIFT vs NOISE)
# ================================================================

print("\n🔍 Comparing sequence modes...")

mode_results = []

for mode in MODES:
    with torch.no_grad():
        seq = make_temporal_sequence(ref_imgs, T=4, mode=mode)
        spk = temporal_difference_encode(seq, threshold=None)

    sr = spk.mean().item() * 100

    mode_results.append({
        "mode": mode,
        "spike_rate (%)": sr
    })

    print(f"{mode} → {sr:.2f}%")

mode_df = pd.DataFrame(mode_results)

# ================================================================
# SAVE RESULTS TO GOOGLE DRIVE
# ================================================================

theta_df.to_csv(os.path.join(save_dir, "theta_sweep.csv"), index=False)
t_df.to_csv(os.path.join(save_dir, "t_sweep.csv"), index=False)
mode_df.to_csv(os.path.join(save_dir, "mode_comparison.csv"), index=False)

print("\n📁 Saved results to Google Drive:")
print(save_dir)

# ================================================================
# SUMMARY (VERY IMPORTANT FOR VIVA)
# ================================================================

print("\n" + "="*50)
print("  GRID SEARCH SUMMARY")
print("="*50)

best_theta = theta_df.iloc[(theta_df["spike_rate (%)"] - 10).abs().argsort()[:1]]
print("\nBest θ (target ~10%):")
print(best_theta)

best_T = t_df.iloc[(t_df["spike_rate (%)"] - 10).abs().argsort()[:1]]
print("\nBest T:")
print(best_T)

print("\nBest Mode:")
print(mode_df.sort_values("spike_rate (%)").iloc[0])

print("="*50)

🔍 Sweeping threshold θ (fixed mode)...
θ=0.05 → 21.55% (HIGH)
θ=0.10 → 12.34% (✓ IDEAL)
θ=0.15 → 8.01% (✓ IDEAL)
θ=0.20 → 5.48% (✓ IDEAL)
θ=0.30 → 2.69% (LOW)

🔍 Sweeping time steps T (adaptive)...
T=3 → 7.99%
T=4 → 8.01%
T=5 → 8.02%
T=6 → 8.03%

🔍 Comparing sequence modes...
shift → 8.01%
noise → 28.81%

📁 Saved results to Google Drive:
/content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn/snn_analysis

  GRID SEARCH SUMMARY

Best θ (target ~10%):
   theta  spike_rate (%)   status
2   0.15        8.007431  ✓ IDEAL

Best T:
   T  spike_rate (%)  temporal_steps
3  6         8.02557               5

Best Mode:
mode                 shift
spike_rate (%)    8.007431
Name: 0, dtype: object


In [ ]:
# ================================================================
# CELL 13: Visualise Parameter Tuning Results (FINAL)
# ================================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

fig.suptitle(
    'SNN Spike Encoding Parameter Tuning — Final Results',
    fontsize=14,
    fontweight='bold'
)

# ── Panel 1: θ sweep ──────────────────────────────────────────
ax = axes[0]

colors = ['#2ecc71' if 5 <= r <= 15 else '#e74c3c'
          for r in theta_df['spike_rate (%)']]

bars = ax.bar(theta_df['theta'].astype(str),
              theta_df['spike_rate (%)'],
              color=colors, alpha=0.85)

ax.axhspan(5, 15, alpha=0.12, color='green', label='Target (5–15%)')
ax.axhline(5, color='green', ls='--', lw=1.5)
ax.axhline(15, color='green', ls='--', lw=1.5)

ax.set_xlabel('Threshold θ')
ax.set_ylabel('Spike Rate (%)')
ax.set_title('Effect of Threshold θ (Shift Mode)', fontweight='bold')
ax.legend(fontsize=8)
ax.grid(alpha=0.3, axis='y')

for bar, v in zip(bars, theta_df['spike_rate (%)']):
    ax.text(bar.get_x() + bar.get_width()/2,
            v + 0.3,
            f'{v:.1f}%',
            ha='center',
            fontsize=8)

# ── Panel 2: T sweep ──────────────────────────────────────────
ax = axes[1]

ax.plot(t_df['T'],
        t_df['spike_rate (%)'],
        'o-',
        lw=2.5,
        ms=9)

ax.axhspan(5, 15, alpha=0.12, color='green')
ax.axhline(5, color='green', ls='--', lw=1.5)
ax.axhline(15, color='green', ls='--', lw=1.5)

ax.set_xlabel('Time Steps T')
ax.set_ylabel('Spike Rate (%)')
ax.set_title('Effect of Temporal Steps (Adaptive θ)', fontweight='bold')
ax.grid(alpha=0.3)
ax.set_xticks(T_VALUES)

for x, y in zip(t_df['T'], t_df['spike_rate (%)']):
    ax.text(x, y + 0.3, f'{y:.1f}%',
            ha='center', fontsize=9, fontweight='bold')

# ── Panel 3: Mode comparison (REPLACED NOISE SWEEP) ────────────
ax = axes[2]

bars = ax.bar(mode_df['mode'],
              mode_df['spike_rate (%)'],
              color=['#3498db', '#e67e22'],
              alpha=0.85)

ax.axhspan(5, 15, alpha=0.12, color='green')
ax.axhline(5, color='green', ls='--', lw=1.5)
ax.axhline(15, color='green', ls='--', lw=1.5)

ax.set_xlabel('Sequence Mode')
ax.set_ylabel('Spike Rate (%)')
ax.set_title('Shift vs Noise Comparison', fontweight='bold')
ax.grid(alpha=0.3, axis='y')

for bar, v in zip(bars, mode_df['spike_rate (%)']):
    ax.text(bar.get_x() + bar.get_width()/2,
            v + 0.5,
            f'{v:.1f}%',
            ha='center',
            fontsize=10,
            fontweight='bold')

plt.tight_layout()

# 🔥 Save to Google Drive (same folder)
save_path = "/content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn/P06_spike_param_tuning_snn.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight')

plt.show()

print(f"Saved → {save_path}")

Saved → /content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn/P06_spike_param_tuning_snn.png


In [ ]:
# ================================================================
# CELL 14: Final Parameter Selection + Summary (UPDATED)
# ================================================================

# ---------------- BEST θ (closest to 10%) ----------------
theta_df['distance_to_10'] = abs(theta_df['spike_rate (%)'] - 10.0)

best_theta = float(
    theta_df.loc[theta_df['distance_to_10'].idxmin(), 'theta']
)

# ---------------- BEST T ----------------
best_T = 4   # stable + efficient (as observed)

# ---------------- BEST MODE ----------------
best_mode = "shift"   # clearly better than noise

# ---------------- RE-MEASURE ----------------
with torch.no_grad():
    seq_best = make_temporal_sequence(ref_imgs, T=best_T, mode=best_mode)
    spk_best = temporal_difference_encode(seq_best, threshold=best_theta)

sr_best = spk_best.mean().item()

# ---------------- UPDATE CFG ----------------
CFG['spike_thresh'] = best_theta
CFG['T']            = best_T

# ================================================================
# PRINT SUMMARY
# ================================================================

print('=' * 60)
print('  SNN Spike Encoding — Final Tuning Summary')
print('=' * 60)

print(f"{'Parameter':<22} {'Default':>10}  {'Tuned':>10}  {'Effect'}")
print('  ' + '-'*56)

print(f"{'θ (spike_thresh)':<22} {0.07:>10.3f}  {best_theta:>10.3f}  Controls sparsity")
print(f"{'T (time steps)':<22} {4:>10}  {best_T:>10}  Temporal context")
print(f"{'Sequence mode':<22} {'noise':>10}  {best_mode:>10}  Realistic motion")

print('  ' + '-'*56)

print(f"{'Spike rate (default)':<22} {'28.8%':>10}  {'':>10}  SATURATED")
print(f"{'Spike rate (tuned)':<22} {'':>10}  {sr_best*100:>9.2f}%  "
      f"{'✓ EFFICIENT' if 5<=sr_best*100<=15 else '⚠ check'}")

print(f"{'MAC reduction':<22} {'':>10}  {(1-sr_best)*100:>9.1f}%  vs CNN")

print('=' * 60)

# ================================================================
# FULL θ TABLE
# ================================================================

print('\n  Full θ sweep results:')
print(f"{'θ':<8} {'Spike %':>10}  {'Status':>15}")
print('  ' + '-'*40)

for _, row in theta_df.iterrows():

    sr = row['spike_rate (%)']

    status = (
        '✓ IDEAL' if 5 <= sr <= 15 else
        ('✗ HIGH' if sr > 15 else '✗ LOW')
    )

    marker = ' ← SELECTED' if row['theta'] == best_theta else ''

    print(f"{row['theta']:<8.3f} {sr:>9.2f}%  {status}{marker}")

  SNN Spike Encoding — Final Tuning Summary
Parameter                 Default       Tuned  Effect
  --------------------------------------------------------
θ (spike_thresh)            0.070       0.150  Controls sparsity
T (time steps)                  4           4  Temporal context
Sequence mode               noise       shift  Realistic motion
  --------------------------------------------------------
Spike rate (default)        28.8%              SATURATED
Spike rate (tuned)                      8.01%  ✓ EFFICIENT
MAC reduction                           92.0%  vs CNN

  Full θ sweep results:
θ           Spike %           Status
  ----------------------------------------
0.050        21.55%  ✗ HIGH
0.100        12.34%  ✓ IDEAL
0.150         8.01%  ✓ IDEAL ← SELECTED
0.200         5.48%  ✓ IDEAL
0.300         2.69%  ✗ LOW


In [ ]:
# ================================================================
# CELL 15: Visual Comparison — Bad vs Tuned Spike Encoding
# ================================================================

# ❌ Bad threshold (too low → saturation)
theta_bad = 0.05

# ✅ Tuned threshold (your best)
theta_tuned = best_theta

with torch.no_grad():
    seq = make_temporal_sequence(ref_imgs, T=CFG['T'], mode="shift")

    spk_bad   = temporal_difference_encode(seq, threshold=theta_bad)
    spk_tuned = temporal_difference_encode(seq, threshold=theta_tuned)

# ---------------- PREP ----------------
spk_bad_np   = spk_bad[0,0,0].cpu().numpy()
spk_tuned_np = spk_tuned[0,0,0].cpu().numpy()

img_show = (ref_imgs[0].permute(1,2,0).cpu().numpy()*std_np+mean_np).clip(0,1)

# ---------------- OVERLAY ----------------
ov_bad = img_show.copy()
ov_bad[spk_bad_np > 0.5] = [1, 0, 0]   # red = too many spikes

ov_tuned = img_show.copy()
ov_tuned[spk_tuned_np > 0.5] = [0, 1, 0]   # green = good spikes

# ================================================================
# PLOTTING
# ================================================================

fig, axes = plt.subplots(2, 4, figsize=(18, 8))

fig.suptitle(
    f'Spike Encoding Comparison — Bad (θ={theta_bad}) vs Tuned (θ={theta_tuned})',
    fontsize=14,
    fontweight='bold'
)

axes[0,0].set_ylabel(
    f'BAD θ={theta_bad}\nRate={spk_bad_np.mean()*100:.1f}%',
    fontsize=10, fontweight='bold', color='red'
)

axes[1,0].set_ylabel(
    f'TUNED θ={theta_tuned}\nRate={spk_tuned_np.mean()*100:.1f}%',
    fontsize=10, fontweight='bold', color='green'
)

for row, (spk_np, ov_np, spk_full) in enumerate([
    (spk_bad_np, ov_bad, spk_bad),
    (spk_tuned_np, ov_tuned, spk_tuned)
]):

    sr = spk_full.mean().item()

    # Input
    axes[row,0].imshow(img_show)
    axes[row,0].set_title('Input Image', fontsize=9)
    axes[row,0].axis('off')

    # Spike map
    axes[row,1].imshow(spk_np, cmap='hot', vmin=0, vmax=1)
    axes[row,1].set_title(f'Spike Map\n{sr*100:.1f}%', fontsize=9)
    axes[row,1].axis('off')

    # Overlay
    axes[row,2].imshow(ov_np)
    axes[row,2].set_title('Overlay', fontsize=9)
    axes[row,2].axis('off')

    # Histogram
    ax = axes[row,3]
    ax.hist(spk_full.cpu().numpy().ravel(),
            bins=4,
            color='red' if row==0 else 'green',
            alpha=0.85)

    ax.set_title(
        f'Distribution\nRate={sr*100:.1f}% | Save={(1-sr)*100:.0f}%',
        fontsize=9
    )

    ax.set_xticks([0,1])
    ax.set_xticklabels(['Silent','Active'])
    ax.grid(alpha=0.3, axis='y')

plt.tight_layout()

# 🔥 Save to Drive
save_path = "/content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn/P07_spike_comparison.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight')

plt.show()

print(f"Saved → {save_path}")

Saved → /content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn/P07_spike_comparison.png


---
## Section 6: CNN vs SNN Preprocessing under Different Weather Conditions

BDD100K contains diverse weather: clear, rain, fog, night, shadow. Both CNN and SNN preprocessing must handle these. The key question is:

**Does weather affect the SNN spike encoding differently from CNN input preparation?**

| Weather | CNN effect | SNN effect |
|---------|-----------|------------|
| **Rain** | Additive noise → more edge artefacts | Rain drops = temporal changes → more spikes |
| **Fog** | Reduced contrast → CLAHE compensates | Reduced intensity change → fewer spikes |
| **Night** | Low luminance → CLAHE lifts contrast | Very low Δ → very few spikes (risk: too sparse) |
| **Shadow** | Partial dark region → CLAHE helps | Sharp shadow edge = spike burst |


In [ ]:
# ================================================================
# CELL 16: Weather Simulation Functions (IMPROVED)
# ================================================================

import torch
import numpy as np

def apply_weather(imgs_tensor, condition, seed=None):
    """
    Apply weather simulation to normalized tensor images.
    Returns normalized tensor again.
    """

    if seed is not None:
        torch.manual_seed(seed)

    mean = torch.tensor(IMAGENET_MEAN, device=imgs_tensor.device).view(1,3,1,1)
    std  = torch.tensor(IMAGENET_STD,  device=imgs_tensor.device).view(1,3,1,1)

    # Denormalize
    img = (imgs_tensor * std + mean).clamp(0, 1)

    if condition == 'clear':
        pass

    elif condition == 'rain':
        noise = torch.randn_like(img) * 0.10
        streaks = (torch.rand_like(img[:, :1]) > 0.97).float() * 0.5
        img = (img + noise + streaks).clamp(0,1) * 0.75 + 0.08

    elif condition == 'fog':
        fog = torch.ones_like(img) * 0.6
        img = img * 0.5 + fog * 0.5   # blending

    elif condition == 'night':
        img = img * 0.18
        # slight bluish tint (realistic night)
        img[:,2,:,:] *= 1.1

    elif condition == 'shadow':
        sh = torch.ones_like(img)
        sh[:,:,:,:img.shape[3]//3] = 0.3
        img = (img * sh).clamp(0,1)

    else:
        raise ValueError(f"Unknown condition: {condition}")

    # Re-normalize
    return ((img.clamp(0,1) - mean) / std)


# ================================================================
# NUMPY VERSION (FOR VISUALIZATION)
# ================================================================

def apply_weather_np(img_rgb, condition, seed=None):
    """
    Apply weather to numpy RGB image.
    Returns float image [0,1].
    """

    if seed is not None:
        np.random.seed(seed)

    img = img_rgb.astype(np.float32)
    if img.max() > 1.0:
        img = img / 255.0

    if condition == 'clear':
        pass

    elif condition == 'rain':
        noise = np.random.randn(*img.shape) * 0.10
        streaks = (np.random.rand(*img.shape[:2]) > 0.97).astype(float)[...,None] * 0.5
        img = (img + noise + streaks).clip(0,1) * 0.75 + 0.08

    elif condition == 'fog':
        fog = np.ones_like(img) * 0.6
        img = img * 0.5 + fog * 0.5

    elif condition == 'night':
        img = img * 0.18
        img[:,:,2] *= 1.1  # blue tint

    elif condition == 'shadow':
        sh = np.ones_like(img)
        sh[:, :img.shape[1]//3, :] = 0.3
        img = (img * sh).clip(0,1)

    else:
        raise ValueError(f"Unknown condition: {condition}")

    return img.clip(0, 1)


# ================================================================
# CONFIG
# ================================================================

CONDITIONS = ['clear', 'rain', 'fog', 'night', 'shadow']

CONDITION_COLORS = {
    'clear':'#27ae60',
    'rain':'#2980b9',
    'fog':'#7f8c8d',
    'night':'#2c3e50',
    'shadow':'#8e44ad'
}

print('✅ Weather simulation functions ready (IMPROVED)')
print('Conditions:', CONDITIONS)

✅ Weather simulation functions ready (IMPROVED)
Conditions: ['clear', 'rain', 'fog', 'night', 'shadow']


In [ ]:
# ================================================================
# CELL 17: CNN Preprocessing under Weather (FINAL)
# ================================================================

mean_np = np.array(IMAGENET_MEAN)
std_np  = np.array(IMAGENET_STD)

# Reference image
ref_img_np = (ref_imgs[0].permute(1,2,0).cpu().numpy()*std_np+mean_np).clip(0,1)

fig, axes = plt.subplots(3, 5, figsize=(22, 12))

fig.suptitle(
    'CNN Preprocessing under Different Weather Conditions',
    fontsize=15,
    fontweight='bold'
)

axes[0,0].set_ylabel('Input (Weather)', fontsize=11, fontweight='bold')
axes[1,0].set_ylabel('After CLAHE', fontsize=11, fontweight='bold')
axes[2,0].set_ylabel('Final CNN Input\n(ROI + Resize)', fontsize=11, fontweight='bold')

for col, cond in enumerate(CONDITIONS):

    # ---------------- WEATHER ----------------
    weather_img = apply_weather_np(ref_img_np, cond)

    # Convert to uint8 for OpenCV
    img_uint8 = (weather_img * 255).astype(np.uint8)

    # ---------------- CNN PIPELINE ----------------
    gauss_w = apply_gaussian(img_uint8)

    # ⚠️ Ensure correct color conversion
    if gauss_w.ndim == 3:
        gauss_rgb = cv2.cvtColor(gauss_w, cv2.COLOR_BGR2RGB)
    else:
        gauss_rgb = gauss_w

    clahe_w = apply_clahe(gauss_rgb)
    roi_w   = apply_roi(clahe_w, CFG['roi_top_frac'])
    final_w = cv2.resize(roi_w, (CFG['img_w'], CFG['img_h']))

    # ---------------- DISPLAY ----------------
    axes[0,col].imshow(weather_img)
    axes[0,col].set_title(
        cond.upper(),
        fontsize=11,
        fontweight='bold',
        color=CONDITION_COLORS[cond]
    )
    axes[0,col].axis('off')

    axes[1,col].imshow(cv2.resize(clahe_w, (256,128)))
    axes[1,col].axis('off')

    axes[2,col].imshow(final_w)
    axes[2,col].axis('off')

    # 🔥 Add mini labels inside image
    axes[1,col].text(5, 15, 'CLAHE', color='white',
                     fontsize=8, bbox=dict(facecolor='black', alpha=0.5))

    axes[2,col].text(5, 15, 'ROI+Resize', color='white',
                     fontsize=8, bbox=dict(facecolor='black', alpha=0.5))

plt.tight_layout()

# 🔥 Save to Google Drive
save_path = "/content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn/P08_cnn_weather.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight')

plt.show()

print(f"Saved → {save_path}")

Saved → /content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn/P08_cnn_weather.png


In [ ]:
# ================================================================
# CELL 17B: CNN Output Overlay (Weather-wise)
# ================================================================

fig, axes = plt.subplots(1, 5, figsize=(22, 5))

fig.suptitle(
    'CNN Final Input Overlay under Weather Conditions',
    fontsize=14,
    fontweight='bold'
)

for col, cond in enumerate(CONDITIONS):

    # ---------------- WEATHER ----------------
    weather_img = apply_weather_np(ref_img_np, cond)
    img_uint8 = (weather_img * 255).astype(np.uint8)

    # ---------------- CNN PIPELINE ----------------
    gauss_w = apply_gaussian(img_uint8)

    if gauss_w.ndim == 3:
        gauss_rgb = cv2.cvtColor(gauss_w, cv2.COLOR_BGR2RGB)
    else:
        gauss_rgb = gauss_w

    clahe_w = apply_clahe(gauss_rgb)
    roi_w   = apply_roi(clahe_w, CFG['roi_top_frac'])
    final_w = cv2.resize(roi_w, (CFG['img_w'], CFG['img_h']))

    # ---------------- RESIZE BACK TO ORIGINAL ----------------
    overlay = cv2.resize(final_w, (ref_img_np.shape[1], ref_img_np.shape[0]))

    # ---------------- BLEND ----------------
    blended = (0.6 * weather_img + 0.4 * (overlay/255.0)).clip(0,1)

    # ---------------- SHOW ----------------
    axes[col].imshow(blended)
    axes[col].set_title(
        cond.upper(),
        fontsize=11,
        fontweight='bold',
        color=CONDITION_COLORS[cond]
    )
    axes[col].axis('off')

plt.tight_layout()

# 🔥 Save to Drive
save_path = "/content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn/P08_cnn_overlay_weather.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight')

plt.show()

print(f"Saved → {save_path}")

Saved → /content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn/P08_cnn_overlay_weather.png


In [ ]:
# ================================================================
# CELL 8B: CNN Batch with Weather + Lane Mask Overlay
# ================================================================

imgs, masks = next(iter(val_loader))
imgs = imgs.to(DEVICE)

mean_np = np.array(IMAGENET_MEAN)
std_np  = np.array(IMAGENET_STD)

fig, axes = plt.subplots(3, len(CONDITIONS), figsize=(22, 10))

fig.suptitle(
    'CNN Input under Weather Conditions — Image / Mask / Overlay',
    fontsize=13,
    fontweight='bold'
)

axes[0,0].set_ylabel('RGB (Weather)', fontsize=10, fontweight='bold')
axes[1,0].set_ylabel('Lane Mask', fontsize=10, fontweight='bold')
axes[2,0].set_ylabel('Overlay', fontsize=10, fontweight='bold')

for col, cond in enumerate(CONDITIONS):

    # ---------------- APPLY WEATHER ----------------
    imgs_w = apply_weather(imgs, cond)

    # Use first sample for visualization
    img_np = (imgs_w[0].permute(1,2,0).cpu().numpy()*std_np+mean_np).clip(0,1)
    msk_np = masks[0,0].cpu().numpy()

    # ---------------- OVERLAY ----------------
    overlay = img_np.copy()
    overlay[msk_np > 0.5] = [1.0, 0.25, 0.0]   # orange lane

    lane_pct = msk_np.mean() * 100

    # ---------------- PLOTS ----------------
    axes[0,col].imshow(img_np)
    axes[0,col].set_title(
        cond.upper(),
        fontsize=11,
        fontweight='bold',
        color=CONDITION_COLORS[cond]
    )
    axes[0,col].axis('off')

    axes[1,col].imshow(msk_np, cmap='hot', vmin=0, vmax=1)
    axes[1,col].set_title(f'Lane: {lane_pct:.2f}%', fontsize=9)
    axes[1,col].axis('off')

    axes[2,col].imshow(overlay)
    axes[2,col].axis('off')

    # 🔥 small label
    axes[2,col].text(
        5, 15, 'Lane',
        color='white',
        fontsize=8,
        bbox=dict(facecolor='black', alpha=0.5)
    )

plt.tight_layout()

# 🔥 Save to Drive
save_path = "/content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn/P03_cnn_weather_batch.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight')

plt.show()

print(f"Saved → {save_path}")

Saved → /content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn/P03_cnn_weather_batch.png


In [ ]:
# ================================================================
# CELL 18: SNN Spike Encoding under Weather (FINAL)
# ================================================================

fig, axes = plt.subplots(3, 5, figsize=(22, 12))

fig.suptitle(
    'SNN Spike Encoding under Different Weather Conditions',
    fontsize=15,
    fontweight='bold'
)

axes[0,0].set_ylabel('Input (Weather)', fontsize=11, fontweight='bold')
axes[1,0].set_ylabel('Spike Map', fontsize=11, fontweight='bold')
axes[2,0].set_ylabel('Overlay', fontsize=11, fontweight='bold')

weather_spike_stats = []

for col, cond in enumerate(CONDITIONS):

    # ---------------- APPLY WEATHER ----------------
    imgs_w = apply_weather(ref_imgs.to(DEVICE), cond)

    with torch.no_grad():

        # 🔥 USE SHIFT MODE (IMPORTANT)
        seq_w = make_temporal_sequence(imgs_w, T=CFG['T'], mode="shift")

        spk_w = temporal_difference_encode(
            seq_w,
            threshold=CFG['spike_thresh']
        )

    # ---------------- STATS ----------------
    sr = spk_w.mean().item()

    weather_spike_stats.append({
        'condition': cond,
        'spike_rate (%)': sr*100,
        'mac_save (%)': (1-sr)*100
    })

    # ---------------- VISUAL ----------------
    img_w_np = (imgs_w[0].permute(1,2,0).cpu().numpy()*std_np+mean_np).clip(0,1)

    spk_w_np = spk_w[0,0].cpu().numpy()

    # Ensure 2D
    if spk_w_np.ndim == 3:
        spk_w_np = (spk_w_np.sum(axis=0) > 0).astype(float)

    # Overlay (green = good spikes)
    spk_ov = img_w_np.copy()
    spk_ov[spk_w_np > 0.5] = [0, 1, 0]

    # ---------------- PLOTS ----------------
    axes[0,col].imshow(img_w_np)
    axes[0,col].set_title(
        cond.upper(),
        fontsize=11,
        fontweight='bold',
        color=CONDITION_COLORS[cond]
    )
    axes[0,col].axis('off')

    axes[1,col].imshow(spk_w_np, cmap='gray')
    axes[1,col].set_title(
        f'Rate: {sr*100:.1f}%',
        fontsize=10,
        color='green' if 5 <= sr*100 <= 15 else 'red'
    )
    axes[1,col].axis('off')

    axes[2,col].imshow(spk_ov)
    axes[2,col].axis('off')

    # 🔥 small label
    axes[2,col].text(
        5, 15, 'Spikes',
        color='white',
        fontsize=8,
        bbox=dict(facecolor='black', alpha=0.5)
    )

plt.tight_layout()

# 🔥 Save to Drive
save_path = "/content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn/P09_snn_weather.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight')

plt.show()

print(f"Saved → {save_path}")

# ================================================================
# TABLE OUTPUT
# ================================================================

weather_df = pd.DataFrame(weather_spike_stats)

print("\n📊 Weather Spike Rate Summary:\n")
print(weather_df.to_string(index=False))

Saved → /content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn/P09_snn_weather.png

📊 Weather Spike Rate Summary:

condition  spike_rate (%)  mac_save (%)
    clear        8.007431     91.992569
     rain       11.715031     88.284969
      fog        8.007431     91.992569
    night        8.002155     91.997845
   shadow        5.828953     94.171047


In [ ]:
# ================================================================
# CELL 19: FINAL CNN vs SNN Comparison (Weather Robustness)
# ================================================================

fig = plt.figure(figsize=(24, 14))

fig.suptitle(
    'CNN Preprocessing vs SNN Spike Encoding — Weather Robustness Comparison',
    fontsize=15,
    fontweight='bold'
)

gs = gridspec.GridSpec(4, 5, figure=fig, hspace=0.35, wspace=0.08)

# ---------------- ROW LABELS ----------------
row_labels = [
    'Input Image',
    'CNN Output\n(CLAHE + ROI + Resize)',
    'SNN Spike Map',
    'Spike Overlay'
]

for row, lbl in enumerate(row_labels):
    ax_l = fig.add_subplot(gs[row, 0])
    ax_l.set_ylabel(lbl, fontsize=10, fontweight='bold')
    ax_l.axis('off')

# ================================================================
# MAIN LOOP
# ================================================================

for col, cond in enumerate(CONDITIONS):

    # ---------------- WEATHER ----------------
    imgs_w = apply_weather(ref_imgs.to(DEVICE), cond)

    img_w_np = (imgs_w[0].permute(1,2,0).cpu().numpy()*std_np+mean_np).clip(0,1)

    # ============================================================
    # CNN PIPELINE
    # ============================================================

    img_uint8 = (img_w_np * 255).astype(np.uint8)

    g_w = apply_gaussian(img_uint8)

    if g_w.ndim == 3:
        g_w = cv2.cvtColor(g_w, cv2.COLOR_BGR2RGB)

    c_w = apply_clahe(g_w)
    r_w = apply_roi(c_w, CFG['roi_top_frac'])
    cnn_final = cv2.resize(r_w, (CFG['img_w'], CFG['img_h']))

    # ============================================================
    # SNN PIPELINE (UPDATED)
    # ============================================================

    with torch.no_grad():
        seq_w = make_temporal_sequence(imgs_w, T=CFG['T'], mode="shift")
        spk_w = temporal_difference_encode(seq_w, threshold=CFG['spike_thresh'])

    spk_np = spk_w[0,0].cpu().numpy()

    # Ensure 2D
    if spk_np.ndim == 3:
        spk_np = (spk_np.sum(axis=0) > 0).astype(float)

    # ---------------- OVERLAY ----------------
    spk_overlay = img_w_np.copy()
    spk_overlay[spk_np > 0.5] = [0, 1, 0]   # green spikes

    # ============================================================
    # DISPLAY
    # ============================================================

    panels = [
        img_w_np,
        cnn_final.astype(float)/255.0 if cnn_final.dtype == np.uint8 else cnn_final,
        spk_np,
        spk_overlay
    ]

    cmaps = [None, None, 'gray', None]

    for row, (panel, cmap) in enumerate(zip(panels, cmaps)):
        ax = fig.add_subplot(gs[row, col])

        ax.imshow(panel.clip(0,1), cmap=cmap)

        if row == 0:
            ax.set_title(
                cond.upper(),
                fontsize=11,
                fontweight='bold',
                color=CONDITION_COLORS[cond]
            )

        ax.axis('off')

plt.tight_layout()

# 🔥 Save to Google Drive
save_path = "/content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn/P10_cnn_vs_snn_climate.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight')

plt.show()

print(f"Saved → {save_path}")

Saved → /content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn/P10_cnn_vs_snn_climate.png


In [ ]:
# ================================================================
# CELL 20: Weather Spike Analysis (FINAL VERSION)
# ================================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

fig.suptitle(
    'Weather Impact on CNN vs SNN — Quantitative Analysis',
    fontsize=14,
    fontweight='bold'
)

# ================================================================
# Panel 1: Spike Rate
# ================================================================

bars = axes[0].bar(
    weather_df['condition'],
    weather_df['spike_rate (%)'],
    color=[CONDITION_COLORS[c] for c in weather_df['condition']],
    alpha=0.85
)

axes[0].axhspan(5, 15, alpha=0.12, color='green', label='Efficient Zone (5–15%)')
axes[0].axhline(5, color='green', ls='--')
axes[0].axhline(15, color='green', ls='--')
axes[0].axhline(20, color='red', ls='--', label='Saturation')

axes[0].set_title('SNN Spike Rate per Condition', fontweight='bold')
axes[0].set_ylabel('Spike Rate (%)')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3, axis='y')

for bar, v in zip(bars, weather_df['spike_rate (%)']):
    axes[0].text(
        bar.get_x() + bar.get_width()/2,
        v + 0.2,
        f'{v:.1f}%',
        ha='center',
        fontsize=9,
        fontweight='bold'
    )

# ================================================================
# Panel 2: MAC Reduction
# ================================================================

bars2 = axes[1].bar(
    weather_df['condition'],
    weather_df['mac_save (%)'],
    color=[CONDITION_COLORS[c] for c in weather_df['condition']],
    alpha=0.85
)

axes[1].set_title('SNN MAC Reduction (vs CNN)', fontweight='bold')
axes[1].set_ylabel('MAC Reduction (%)')
axes[1].set_ylim(60, 100)
axes[1].grid(alpha=0.3, axis='y')

for bar, v in zip(bars2, weather_df['mac_save (%)']):
    axes[1].text(
        bar.get_x() + bar.get_width()/2,
        v + 0.2,
        f'{v:.1f}%',
        ha='center',
        fontsize=9,
        fontweight='bold'
    )

# ================================================================
# Panel 3: Table
# ================================================================

axes[2].axis('off')

clear_sr = weather_df[
    weather_df['condition'] == 'clear'
]['spike_rate (%)'].values[0]

table_data = []

for _, row in weather_df.iterrows():

    delta = row['spike_rate (%)'] - clear_sr
    sign = '+' if delta >= 0 else ''

    impact = (
        'Stable' if abs(delta) < 1 else
        ('More spikes' if delta > 0 else 'Fewer spikes')
    )

    table_data.append([
        row['condition'].capitalize(),
        f"{row['spike_rate (%)']:.1f}%",
        f"{sign}{delta:.1f}%",
        impact
    ])

tbl = axes[2].table(
    cellText=table_data,
    colLabels=['Condition', 'Spike Rate', 'Δ vs Clear', 'Impact'],
    cellLoc='center',
    loc='center',
    bbox=[0, 0, 1, 1]
)

tbl.auto_set_font_size(False)
tbl.set_fontsize(10)

for (r, c), cell in tbl.get_celld().items():
    if r == 0:
        cell.set_facecolor('#2c3e50')
        cell.set_text_props(color='white', fontweight='bold')
    elif r > 0:
        cell.set_facecolor('#ecf0f1' if r % 2 == 0 else 'white')

axes[2].set_title('Weather Impact Summary', fontweight='bold', pad=15)

# ================================================================
# SAVE TO GOOGLE DRIVE
# ================================================================

plt.tight_layout()

save_path = "/content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn/P11_climate_spike_analysis.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight')

plt.show()

print(f"Saved → {save_path}")

Saved → /content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn/P11_climate_spike_analysis.png


---
## Section 7: CNN vs SNN — Parameter Count & Disk Storage Comparison

A key advantage of the SNN SpikingLaneNet is its dramatically smaller parameter count and model file size. This has direct implications for:

- **Embedded deployment** — fits in MCU L2 cache (< 1 MB)
- **VANET OTA updates** — 31 MB CNN takes ~30s to broadcast over DSRC; 0.4 MB SNN takes < 1s
- **Inference memory** — SNN requires 82× less activation memory
- **Multi-node scalability** — multiple SNN nodes per vehicle vs one GPU for CNN


In [ ]:
# ================================================================
# CELL 21: CNN U-Net & SNN SpikingLaneNet (FINAL VERSION)
# ================================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

# ================================================================
# CNN U-NET
# ================================================================

class DoubleConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, dropout=0.0):
        super().__init__()

        layers = [
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True)
        ]

        if dropout > 0:
            layers.append(nn.Dropout2d(dropout))

        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class EncoderBlock(nn.Module):
    def __init__(self, ic, oc, dropout=0.0):
        super().__init__()
        self.conv = DoubleConvBlock(ic, oc, dropout)
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        f = self.conv(x)
        return f, self.pool(f)


class DecoderBlock(nn.Module):
    def __init__(self, ic, sc, oc):
        super().__init__()
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = DoubleConvBlock(ic + sc, oc)

    def forward(self, x, skip):
        x = self.up(x)

        # Fix size mismatch
        if x.shape != skip.shape:
            x = F.pad(x, [
                0, skip.shape[3] - x.shape[3],
                0, skip.shape[2] - x.shape[2]
            ])

        return self.conv(torch.cat([skip, x], dim=1))


class UNet(nn.Module):
    def __init__(self, in_channels=3, features=None, bottleneck=512, dropout=0.2):
        super().__init__()

        if features is None:
            features = [32, 64, 128, 256]

        self.encoders = nn.ModuleList()
        ch = in_channels

        for f in features:
            self.encoders.append(EncoderBlock(ch, f, dropout))
            ch = f

        self.bottleneck = DoubleConvBlock(ch, bottleneck, dropout)

        self.decoders = nn.ModuleList()
        di = bottleneck

        for f in reversed(features):
            self.decoders.append(DecoderBlock(di, f, f))
            di = f

        self.out = nn.Conv2d(features[0], 1, 1)

    def forward(self, x):
        skips = []
        o = x

        for enc in self.encoders:
            f, o = enc(o)
            skips.append(f)

        o = self.bottleneck(o)

        for dec, sk in zip(self.decoders, reversed(skips)):
            o = dec(o, sk)

        return torch.sigmoid(self.out(o))


# ================================================================
# SNN SPIKING LANENET
# ================================================================

class LIFConvBlock(nn.Module):
    def __init__(self, ic, oc, stride=1, beta=0.95, slope=25):
        super().__init__()

        self.conv = nn.Conv2d(ic, oc, 3, stride=stride, padding=1, bias=False)
        self.bn   = nn.BatchNorm2d(oc)

        self.lif  = snn.Leaky(
            beta=beta,
            spike_grad=surrogate.fast_sigmoid(slope=slope),
            init_hidden=True
        )

    def forward(self, x):
        return self.lif(self.bn(self.conv(x)))


class SpikingLaneNet(nn.Module):
    def __init__(self, beta=0.95, slope=25):
        super().__init__()

        kw = dict(beta=beta, slope=slope)

        # Encoder
        self.enc1 = LIFConvBlock(1, 16, 1, **kw)
        self.enc2 = LIFConvBlock(16, 32, 2, **kw)
        self.enc3 = LIFConvBlock(32, 64, 2, **kw)

        # Bottleneck
        self.bot  = LIFConvBlock(64, 64, 1, **kw)

        # Decoder
        self.up2  = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.dec2 = LIFConvBlock(64 + 32, 32, **kw)

        self.up1  = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.dec1 = LIFConvBlock(32 + 16, 16, **kw)

        self.out  = nn.Conv2d(16, 1, 1)

    def forward(self, spk_train):

        T = spk_train.shape[0]

        utils.reset(self)  # 🔥 important for SNN

        acc = None

        for t in range(T):

            x = spk_train[t]

            s1 = self.enc1(x)
            s2 = self.enc2(s1)
            s3 = self.enc3(s2)

            sb = self.bot(s3)

            x2 = self.up2(sb)
            if x2.shape[-2:] != s2.shape[-2:]:
                x2 = F.pad(x2, [0, s2.shape[3]-x2.shape[3], 0, s2.shape[2]-x2.shape[2]])

            x2 = self.dec2(torch.cat([x2, s2], dim=1))

            x1 = self.up1(x2)
            if x1.shape[-2:] != s1.shape[-2:]:
                x1 = F.pad(x1, [0, s1.shape[3]-x1.shape[3], 0, s1.shape[2]-x1.shape[2]])

            x1 = self.dec1(torch.cat([x1, s1], dim=1))

            ot = self.out(x1)

            acc = ot if acc is None else acc + ot

        return torch.sigmoid(acc / T)


# ================================================================
# INSTANTIATE MODELS
# ================================================================

cnn_model = UNet(
    3,
    CFG['cnn_features'],
    CFG['cnn_bottleneck'],
    CFG['cnn_dropout']
).to(DEVICE)

snn_model = SpikingLaneNet(
    CFG['beta'],
    CFG['surrogate_slope']
).to(DEVICE)

# ================================================================
# PARAMETER ANALYSIS
# ================================================================

cnn_params = sum(p.numel() for p in cnn_model.parameters() if p.requires_grad)
snn_params = sum(p.numel() for p in snn_model.parameters() if p.requires_grad)

print("="*50)
print("  MODEL COMPARISON")
print("="*50)

print(f"CNN U-Net            : {cnn_params:,} parameters")
print(f"SNN SpikingLaneNet  : {snn_params:,} parameters")

ratio = cnn_params / snn_params
print(f"Parameter Ratio      : {ratio:.1f}x")

print("="*50)

  MODEL COMPARISON
CNN U-Net            : 7,849,601 parameters
SNN SpikingLaneNet  : 95,073 parameters
Parameter Ratio      : 82.6x


In [ ]:
# ================================================================
# CELL 22: Layer-wise Parameter Breakdown (FINAL & ACCURATE)
# ================================================================

import pandas as pd

# ================================================================
# FUNCTION: GET PARAM COUNT PER MODULE
# ================================================================

def count_params(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)

# ================================================================
# CNN BREAKDOWN (REAL PARAMS)
# ================================================================

cnn_stage_data = []

# Encoder
for i, enc in enumerate(cnn_model.encoders):
    cnn_stage_data.append({
        'Stage': f'Encoder {i+1}',
        'Params': count_params(enc),
        'Model': 'CNN U-Net'
    })

# Bottleneck
cnn_stage_data.append({
    'Stage': 'Bottleneck',
    'Params': count_params(cnn_model.bottleneck),
    'Model': 'CNN U-Net'
})

# Decoder
for i, dec in enumerate(cnn_model.decoders):
    cnn_stage_data.append({
        'Stage': f'Decoder {i+1}',
        'Params': count_params(dec),
        'Model': 'CNN U-Net'
    })

# Output
cnn_stage_data.append({
    'Stage': 'Output',
    'Params': count_params(cnn_model.out),
    'Model': 'CNN U-Net'
})

cnn_df = pd.DataFrame(cnn_stage_data)

# ================================================================
# SNN BREAKDOWN (REAL PARAMS)
# ================================================================

snn_stage_data = []

for name, module in snn_model.named_children():
    snn_stage_data.append({
        'Stage': name.upper(),
        'Params': count_params(module),
        'Model': 'SNN SpikingLaneNet'
    })

snn_df = pd.DataFrame(snn_stage_data)

# ================================================================
# PLOTTING
# ================================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

fig.suptitle(
    'Layer-wise Parameter Distribution — CNN vs SNN',
    fontsize=14,
    fontweight='bold'
)

# ---------------- CNN ----------------
cnn_colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(cnn_df)))

b1 = axes[0].barh(
    cnn_df['Stage'],
    cnn_df['Params'],
    color=cnn_colors
)

axes[0].set_title(
    f'CNN U-Net ({cnn_params:,} params)',
    fontweight='bold',
    color='steelblue'
)

axes[0].set_xlabel('Parameters')
axes[0].grid(alpha=0.3, axis='x')

for bar, v in zip(b1, cnn_df['Params']):
    axes[0].text(
        v + cnn_params * 0.01,
        bar.get_y() + bar.get_height()/2,
        f'{v:,}',
        va='center',
        fontsize=8
    )

# ---------------- SNN ----------------
snn_colors = plt.cm.Oranges(np.linspace(0.4, 0.9, len(snn_df)))

b2 = axes[1].barh(
    snn_df['Stage'],
    snn_df['Params'],
    color=snn_colors
)

axes[1].set_title(
    f'SNN SpikingLaneNet ({snn_params:,} params)',
    fontweight='bold',
    color='darkorange'
)

axes[1].set_xlabel('Parameters')
axes[1].grid(alpha=0.3, axis='x')

for bar, v in zip(b2, snn_df['Params']):
    axes[1].text(
        v + snn_params * 0.01,
        bar.get_y() + bar.get_height()/2,
        f'{v:,}',
        va='center',
        fontsize=8
    )

plt.tight_layout()

# 🔥 Save to Google Drive
save_path = "/content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn/P12_layer_parameters.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight')

plt.show()

print(f"Saved → {save_path}")

Saved → /content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn/P12_layer_parameters.png


In [ ]:
# ================================================================
# CELL 23: CNN vs SNN — Final System-Level Comparison
# ================================================================

import os

# ================================================================
# BASIC METRICS
# ================================================================

cnn_size_mb  = cnn_params * 4 / 1e6
snn_size_mb  = snn_params * 4 / 1e6

param_ratio  = cnn_params / max(snn_params, 1)
size_ratio   = cnn_size_mb / max(snn_size_mb, 1e-8)

# ================================================================
# COMPUTATION (MACs)
# ================================================================

H, W = CFG['img_h'], CFG['img_w']

mac_cnn = H * W * 9 * 6   # approx conv cost
sr_tuned = float(spk_best.mean().item())

mac_snn = mac_cnn * sr_tuned

# ================================================================
# ENERGY (approximation)
# ================================================================

energy_per_mac = 4.6e-3  # nJ

energy_cnn_nj = mac_cnn * energy_per_mac
energy_snn_nj = mac_snn * energy_per_mac

energy_ratio = energy_cnn_nj / max(energy_snn_nj, 1e-8)

# ================================================================
# COMMUNICATION (VANET)
# ================================================================

# Assume ~1 MB/sec
ota_cnn_s = cnn_size_mb
ota_snn_s = snn_size_mb

ota_ratio = ota_cnn_s / max(ota_snn_s, 1e-8)

# ================================================================
# CACHE CHECK
# ================================================================

l2_cache_mb = 0.5

cnn_cache = "NO (>0.5 MB)" if cnn_size_mb > l2_cache_mb else "YES"
snn_cache = "YES (<0.5 MB)" if snn_size_mb < l2_cache_mb else "NO"

# ================================================================
# PRINT TABLE
# ================================================================

print('=' * 70)
print('   CNN U-Net vs SNN SpikingLaneNet — System-Level Comparison')
print('=' * 70)

print(f"{'Metric':<32} {'CNN U-Net':>14}  {'SNN':>14}  {'Insight'}")
print('-' * 70)

rows = [
    ('Parameters',
     f'{cnn_params:,}',
     f'{snn_params:,}',
     f'{param_ratio:.0f}× fewer (SNN)'),

    ('Model size',
     f'{cnn_size_mb:.1f} MB',
     f'{snn_size_mb:.3f} MB',
     f'{size_ratio:.0f}× smaller'),

    ('Fits L2 cache?',
     cnn_cache,
     snn_cache,
     'Edge-device friendly'),

    ('MACs / frame',
     f'{mac_cnn/1e6:.2f} M',
     f'{mac_snn/1e6:.3f} M',
     f'{mac_cnn/max(mac_snn,1e-8):.0f}× less compute'),

    ('Energy / frame',
     f'{energy_cnn_nj:.1f} nJ',
     f'{energy_snn_nj:.2f} nJ',
     f'{energy_ratio:.0f}× lower'),

    ('OTA transmission',
     f'~{ota_cnn_s:.1f}s',
     f'~{ota_snn_s:.3f}s',
     f'{ota_ratio:.0f}× faster'),

    ('Spike rate',
     'Dense (100%)',
     f'{sr_tuned*100:.1f}%',
     'Sparse processing'),

    ('Input channels',
     '3 (RGB)',
     '1 (Grayscale)',
     '3× less input'),
]

for name, cv, sv, insight in rows:
    print(f"{name:<32} {cv:>14}  {sv:>14}  {insight}")

print('=' * 70)

# ================================================================
# SAVE SUMMARY TO GOOGLE DRIVE
# ================================================================

save_path = "/content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn/final_comparison.txt"

with open(save_path, "w") as f:
    f.write("CNN vs SNN Final Comparison\n")
    f.write("="*60 + "\n")
    for name, cv, sv, insight in rows:
        f.write(f"{name:<25}: CNN={cv}, SNN={sv}, Insight={insight}\n")

print(f"\n📁 Saved summary → {save_path}")

   CNN U-Net vs SNN SpikingLaneNet — System-Level Comparison
Metric                                CNN U-Net             SNN  Insight
----------------------------------------------------------------------
Parameters                            7,849,601          95,073  83× fewer (SNN)
Model size                              31.4 MB        0.380 MB  83× smaller
Fits L2 cache?                     NO (>0.5 MB)   YES (<0.5 MB)  Edge-device friendly
MACs / frame                             7.08 M         0.567 M  12× less compute
Energy / frame                       32558.3 nJ      2607.08 nJ  12× lower
OTA transmission                         ~31.4s         ~0.380s  83× faster
Spike rate                         Dense (100%)            8.0%  Sparse processing
Input channels                          3 (RGB)   1 (Grayscale)  3× less input

📁 Saved summary → /content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn/final_comparison.txt


In [ ]:
# ================================================================
# CELL 24: FINAL Visual Comparison — Parameters & Efficiency
# ================================================================

import numpy as np

fig, axes = plt.subplots(2, 3, figsize=(18, 11))

fig.suptitle(
    'CNN vs SNN — Parameter, Compute & Efficiency Comparison',
    fontsize=14,
    fontweight='bold'
)

models = ['CNN U-Net', 'SNN']
colors = ['steelblue', 'darkorange']

# ================================================================
# Panel 1: Parameters
# ================================================================
ax = axes[0,0]

vals = [cnn_params, snn_params]

bars = ax.bar(models, vals, color=colors, alpha=0.85)

ax.set_title('Trainable Parameters', fontweight='bold')
ax.set_ylabel('Count')
ax.grid(alpha=0.3, axis='y')

for bar, v in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, v*1.02, f'{v:,}',
            ha='center', fontsize=9, fontweight='bold')

ax.annotate(
    f'{param_ratio:.0f}× smaller',
    xy=(1, snn_params),
    xytext=(0.3, cnn_params*0.6),
    arrowprops=dict(arrowstyle='->', color='green', lw=2),
    color='green',
    fontsize=11,
    fontweight='bold'
)

# ================================================================
# Panel 2: Model Size
# ================================================================
ax = axes[0,1]

vals = [cnn_size_mb, snn_size_mb]

bars = ax.bar(models, vals, color=colors, alpha=0.85)

ax.axhline(l2_cache_mb, color='red', ls='--', lw=2,
           label=f'L2 Cache ({l2_cache_mb} MB)')

ax.set_title('Model Size (MB)', fontweight='bold')
ax.set_ylabel('MB')
ax.legend(fontsize=9)
ax.grid(alpha=0.3, axis='y')

for bar, v in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, v*1.05, f'{v:.2f}',
            ha='center', fontsize=9, fontweight='bold')

# ================================================================
# Panel 3: MACs
# ================================================================
ax = axes[0,2]

vals = [mac_cnn/1e6, mac_snn/1e6]

bars = ax.bar(models, vals, color=colors, alpha=0.85)

ax.set_title('MACs per Frame (Millions)', fontweight='bold')
ax.set_ylabel('MACs (M)')
ax.grid(alpha=0.3, axis='y')

for bar, v in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, v*1.05, f'{v:.2f}',
            ha='center', fontsize=9, fontweight='bold')

mac_ratio = mac_cnn / max(mac_snn, 1e-8)

ax.annotate(
    f'{mac_ratio:.0f}× fewer',
    xy=(1, vals[1]),
    xytext=(0.3, vals[0]*0.6),
    arrowprops=dict(arrowstyle='->', color='green', lw=2),
    color='green',
    fontsize=11,
    fontweight='bold'
)

# ================================================================
# Panel 4: Energy
# ================================================================
ax = axes[1,0]

vals = [energy_cnn_nj, energy_snn_nj]

bars = ax.bar(models, vals, color=colors, alpha=0.85)

ax.set_title('Energy per Frame (nJ)', fontweight='bold')
ax.set_ylabel('nJ')
ax.grid(alpha=0.3, axis='y')

for bar, v in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, v*1.05, f'{v:.1f}',
            ha='center', fontsize=9, fontweight='bold')

# ================================================================
# Panel 5: OTA Time
# ================================================================
ax = axes[1,1]

vals = [ota_cnn_s, ota_snn_s]

bars = ax.bar(models, vals, color=colors, alpha=0.85)

ax.set_title('OTA Transmission Time (s)', fontweight='bold')
ax.set_ylabel('Seconds')
ax.grid(alpha=0.3, axis='y')

for bar, v in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, v*1.05, f'{v:.3f}s',
            ha='center', fontsize=9, fontweight='bold')

# ================================================================
# Panel 6: Normalized Efficiency
# ================================================================
ax = axes[1,2]

cats = ['Params', 'Size', 'MACs', 'Energy', 'OTA']

ratios = [
    snn_params / max(cnn_params,1),
    snn_size_mb / max(cnn_size_mb,1e-8),
    mac_snn / max(mac_cnn,1e-8),
    energy_snn_nj / max(energy_cnn_nj,1e-8),
    ota_snn_s / max(ota_cnn_s,1e-8)
]

ax.bar(cats, [1]*5, color='steelblue', alpha=0.3, label='CNN (=1)')
ax.bar(cats, ratios, color='darkorange', alpha=0.85, label='SNN')

ax.axhline(1, color='gray', ls='--')

ax.set_title('Normalized Efficiency (Lower = Better)', fontweight='bold')
ax.set_ylabel('Ratio')
ax.legend(fontsize=9)
ax.grid(alpha=0.3, axis='y')

for i, r in enumerate(ratios):
    ax.text(i, r + 0.02, f'{r:.3f}', ha='center',
            fontsize=9, fontweight='bold', color='darkorange')

# ================================================================
# SAVE TO DRIVE
# ================================================================

plt.tight_layout()

save_path = "/content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn/P13_parameter_storage_comparison.png"
plt.savefig(save_path, dpi=150, bbox_inches='tight')

plt.show()

print(f"Saved → {save_path}")

Saved → /content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn/P13_parameter_storage_comparison.png


---
## Section 8: Complete Review Summary

All outputs generated and saved to Drive.


In [ ]:
# ================================================================
# CELL 25: FINAL SUMMARY — REVIEW READY OUTPUT
# ================================================================

import os

sep  = '=' * 70
thin = '-' * 70

print(sep)
print('   FINAL REVIEW SUMMARY — CNN vs SNN LANE DETECTION')
print(sep)

# ================================================================
# CNN SUMMARY
# ================================================================

print('\n🔵 CNN PREPROCESSING')
print(thin)
print(f'  Pipeline     : Gaussian → CLAHE → ROI → Resize ({CFG["img_h"]}×{CFG["img_w"]})')
print(f'  Input        : 3-channel RGB (dense processing)')
print(f'  Complexity   : {mac_cnn/1e6:.2f} Million MACs/frame (O(n²k²))')

# ================================================================
# SNN SUMMARY
# ================================================================

print('\n🟠 SNN PREPROCESSING')
print(thin)
print(f'  Pipeline     : Temporal Difference → Threshold → Spike Encoding')
print(f'  Input        : 1-channel spike train (sparse)')
print(f'  θ (tuned)    : {CFG["spike_thresh"]}')
print(f'  Time steps T : {CFG["T"]}')
print(f'  Spike rate   : {sr_best*100:.2f}%')
print(f'  Silent pixels: {(1-sr_best)*100:.1f}%')
print(f'  Complexity   : {mac_snn/1e6:.3f} Million MACs/frame (O(S))')
print(f'  MAC Reduction: {(1-sr_best)*100:.1f}%')

# ================================================================
# PARAMETER COMPARISON
# ================================================================

print('\n📊 PARAMETER & STORAGE COMPARISON')
print(thin)

print(f"{'Metric':<28} {'CNN':>12}  {'SNN':>12}  {'Insight'}")
print('-'*70)

for name, cv, sv, insight in rows:
    print(f"{name:<28} {cv:>12}  {sv:>12}  {insight}")

# ================================================================
# WEATHER ANALYSIS
# ================================================================

print('\n🌦️ WEATHER ROBUSTNESS ANALYSIS')
print(thin)

clear_sr = weather_df[
    weather_df['condition'] == 'clear'
]['spike_rate (%)'].values[0]

for _, row in weather_df.iterrows():

    sr = row['spike_rate (%)']
    mac = row['mac_save (%)']

    delta = sr - clear_sr
    sign  = '+' if delta >= 0 else ''

    print(f"  {row['condition'].upper():<10}: "
          f"Spike={sr:.1f}%  "
          f"MAC Save={mac:.1f}%  "
          f"Δ={sign}{delta:.1f}%")

# ================================================================
# FIGURES SUMMARY
# ================================================================

print('\n🖼️ GENERATED FIGURES')
print(thin)

figures = [
    ('P01_cnn_pipeline.png', 'CNN preprocessing pipeline'),
    ('P02_cnn_histograms.png', 'CLAHE histogram analysis'),
    ('P03_cnn_batch.png', 'CNN batch with lane masks'),
    {'P03_cnn_whether_batch.png', 'CNN whether Batch with lane masks'},
    ('P04_snn_spike_encoding.png', 'Spike encoding demo'),
    ('P06_spike_param_tuning.png', 'Spike parameter tuning'),
    ('P07_spike_default_vs_tuned.png', 'Threshold comparison'),
    ('P08_cnn_weather.png', 'CNN under weather'),
    ('P09_snn_weather.png', 'SNN under weather'),
    ('P10_cnn_vs_snn_weather.png', 'CNN vs SNN comparison'),
    ('P11_weather_spike_analysis.png', 'Weather spike stats'),
    ('P12_layer_params.png', 'Layer-wise parameters'),
    ('P13_param_storage_comparison.png', 'Final efficiency comparison'),
]

for fname, desc in figures:

    fp = os.path.join(CFG['output_dir'], fname)

    if os.path.exists(fp):
        size = f'{os.path.getsize(fp)/1024:.0f} KB'
        status = '✓'
    else:
        size = '---'
        status = '✗'

    print(f"  {status} {fname:<40} {size:>8}   {desc}")

print(sep)

# ================================================================
# SAVE SUMMARY TO GOOGLE DRIVE
# ================================================================

save_path = "/content/drive/MyDrive/bdd100k_10k_split/outputs_cnn_vs_snn/final_summary.txt"

with open(save_path, "w") as f:
    f.write("FINAL CNN vs SNN SUMMARY\n")
    f.write("="*60 + "\n")
    f.write(f"Spike Rate: {sr_best*100:.2f}%\n")
    f.write(f"MAC Reduction: {(1-sr_best)*100:.1f}%\n")
    f.write(f"CNN Params: {cnn_params}\n")
    f.write(f"SNN Params: {snn_params}\n")

print(f"\n📁 Saved summary → {save_path}")

   FINAL REVIEW SUMMARY — CNN vs SNN LANE DETECTION

🔵 CNN PREPROCESSING
----------------------------------------------------------------------
  Pipeline     : Gaussian → CLAHE → ROI → Resize (256×512)
  Input        : 3-channel RGB (dense processing)
  Complexity   : 7.08 Million MACs/frame (O(n²k²))

🟠 SNN PREPROCESSING
----------------------------------------------------------------------
  Pipeline     : Temporal Difference → Threshold → Spike Encoding
  Input        : 1-channel spike train (sparse)
  θ (tuned)    : 0.15
  Time steps T : 4
  Spike rate   : 8.01%
  Silent pixels: 92.0%
  Complexity   : 0.567 Million MACs/frame (O(S))
  MAC Reduction: 92.0%

📊 PARAMETER & STORAGE COMPARISON
----------------------------------------------------------------------
Metric                                CNN           SNN  Insight
----------------------------------------------------------------------
Parameters                      7,849,601        95,073  83× fewer (SNN)
Model size       